In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df = pickle.load(open('../data/processed_df.pkl', 'rb'))

In [3]:
df.head()

,item_id,consumer_id,consumer_location,country,producer_id,producer_session_id,producer_location,producer_country,item_type,item_url,title,text_description,language
0,-3499919498720038879,-8845298781299428018,NaN,NaN,-1032019229384696495,-7696592431575292648,NaN,NaN,HTML,http://techcrunch.com/2016/06/07/hiri/,Hiri wants to fix the workplace email problem,Hiri is the latest startup trying to fix email...,en
1,8890720798209849691,-1032019229384696495,NY,US,1895326251577378793,-1847389231177111235,NaN,NaN,HTML,https://www.nngroup.com/articles/top-intranet-...,Top 10 Intranet Trends of 2016,"Summary: Hero images, carousels, fat footers, ...",en
2,310515487419366995,-1130272294246983140,NaN,NaN,-5527145562136413747,3192880191028684629,NaN,NaN,HTML,http://blog.runrun.it/erros-de-portugues-em-e-...,71 erros de português que precisam sumir dos s...,Escrever um e-mail não deveria ser uma coisa t...,pt
3,310515487419366995,344280948527967603,NaN,NaN,-5527145562136413747,3192880191028684629,NaN,NaN,HTML,http://blog.runrun.it/erros-de-portugues-em-e-...,71 erros de português que precisam sumir dos s...,Escrever um e-mail não deveria ser uma coisa t...,pt
4,-7820640624231356730,-445337111692715325,NaN,NaN,-1032019229384696495,3621737643587579081,NaN,NaN,HTML,http://www.fastcompany.com/3058876/your-most-p...,How This Googler Redesigned The Workweek,Jake Knapp has always been concerned about the...,en


## Prepare the Interaction Data

In [4]:
# Drop duplicate item_ids, keeping the first occurrence
df_unique_items = df.drop_duplicates(subset='item_id', keep='first')

# Create the dictionary
item_info_dict = {
    row['item_id']: {
        "title": row['title'],
        "text_description": row['text_description']
    }
    for _, row in df_unique_items.iterrows()
}
pickle.dump(item_info_dict, open('../data/item_info_dict.pkl', 'wb'))

In [5]:
# Create implicit feedback (all interactions as 1)
df_interactions = df[['consumer_id', 'item_id']].copy()
df_interactions['interaction'] = 1

In [6]:
df_interactions.head()

,consumer_id,item_id,interaction
0,-8845298781299428018,-3499919498720038879,1
1,-1032019229384696495,8890720798209849691,1
2,-1130272294246983140,310515487419366995,1
3,344280948527967603,310515487419366995,1
4,-445337111692715325,-7820640624231356730,1


In [7]:
df_interactions.duplicated().sum()

np.int64(11189)

In [8]:
df = df_interactions.drop_duplicates()

In [9]:
df = df.reset_index(drop=True, inplace=True)

## Create a User-Item Matrix

In [10]:
user_item_matrix = df_interactions.pivot_table(
    index='consumer_id',
    columns='item_id',
    values='interaction',
    fill_value=0
)

## Perform Collaborative Filtering - User-User similarity

In [11]:
user_similarity = cosine_similarity(user_item_matrix)

In [12]:
user_similarity

array([[1.        , 0.        , 0.        , ..., 0.04402255, 0.04075696,
        0.07685206],
       [0.        , 1.        , 0.16903085, ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.16903085, 1.        , ..., 0.21821789, 0.10101525,
        0.        ],
       ...,
       [0.04402255, 0.        , 0.21821789, ..., 1.        , 0.07715167,
        0.        ],
       [0.04075696, 0.        , 0.10101525, ..., 0.07715167, 1.        ,
        0.        ],
       [0.07685206, 0.        , 0.        , ..., 0.        , 0.        ,
        1.        ]], shape=(1895, 1895))

In [13]:
user_similarity_df = pd.DataFrame(user_similarity, 
                                   index=user_item_matrix.index, 
                                   columns=user_item_matrix.index)

In [14]:
user_similarity_df.head()

consumer_id,-9223121837663643404,-9212075797126931087,-9207251133131336884,-9199575329909162940,-9196668942822132778,-9188188261933657343,-9172914609055320039,-9156344805277471150,-9150583489352258206,-9120685872592674274,...,9128735267433534024,9135582630122950040,9137372837662939523,9148269800512008413,9152852494775096912,9165571805999894845,9187866633451383747,9191849144618614467,9199170757466086545,9210530975708218054
consumer_id,,,,,,,,,,,,,,,,,,,,,
-9223121837663643404,1.00000,0.000000,0.000000,0.09196,0.0,0.0,0.093386,0.000000,0.0,0.031129,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.036986,0.044023,0.040757,0.076852
-9212075797126931087,0.00000,1.000000,0.169031,0.00000,0.0,0.0,0.000000,0.119523,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000
-9207251133131336884,0.00000,0.169031,1.000000,0.00000,0.0,0.0,0.000000,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.218218,0.101015,0.000000
-9199575329909162940,0.09196,0.000000,0.000000,1.00000,0.0,0.0,0.123091,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000
-9196668942822132778,0.00000,0.000000,0.000000,0.00000,1.0,0.0,0.000000,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.169031,0.000000,0.000000,0.000000,0.047619


In [15]:
pickle.dump(user_similarity_df, open('../data/user_similarity_df.pkl', 'wb'))
pickle.dump(user_item_matrix, open('../data/user_item_matrix.pkl', 'wb'))

In [16]:
# def recommend_items(user_id, user_similarity_df, user_item_matrix, n=5):
#     # Step 1: Get similar users and their similarity scores
#     if user_id not in user_similarity_df.index:
#         return []

#     similar_users = user_similarity_df[user_id].sort_values(ascending=False).drop(user_id)

#     # Step 2: Get items interacted with by similar users
#     interacted_items = set(user_item_matrix.loc[user_id][user_item_matrix.loc[user_id] > 0].index)
#     scores = {}

#     for sim_user_id, similarity in similar_users.items():
#         sim_user_items = user_item_matrix.loc[sim_user_id]
#         for item_id, interacted in sim_user_items.items():
#             if interacted > 0 and item_id not in interacted_items:
#                 if item_id not in scores:
#                     scores[item_id] = 0
#                 scores[item_id] += similarity  # Weighted by similarity

#     # Step 3: Sort items by score and return top N
#     top_items = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:n]
#     return top_items  # Returns (item_id, score)

def recommend_items(user_id, user_similarity_df, user_item_matrix, item_info_dict, n=5):
    # Step 1: Get similar users and their similarity scores
    if user_id not in user_similarity_df.index:
        return []

    similar_users = user_similarity_df[user_id].sort_values(ascending=False).drop(user_id)

    # Step 2: Get items the user already interacted with
    interacted_items = set(user_item_matrix.loc[user_id][user_item_matrix.loc[user_id] > 0].index)

    scores = {}
    for sim_user_id, similarity in similar_users.items():
        sim_user_items = user_item_matrix.loc[sim_user_id]
        for item_id, interacted in sim_user_items.items():
            if interacted > 0 and item_id not in interacted_items:
                if item_id not in scores:
                    scores[item_id] = 0
                scores[item_id] += similarity  # Weighted by similarity

    # Step 3: Sort items by score and return top N
    top_items = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:n]

    # Step 4: Map item_id to title and text_description
    recommendations = []
    for item_id, score in top_items:
        info = item_info_dict.get(item_id, {"title": "Unknown", "text_description": "No description available."})
        recommendations.append({
            "item_id": item_id,
            "title": info["title"],
            "text_description": info["text_description"],
            "score": score
        })

    return recommendations


In [17]:
recommend_items(-8845298781299428018, user_similarity_df, user_item_matrix, item_info_dict, n=5)

[{'item_id': -4029704725707465084,
  'title': 'Former Google career coach shares a visual trick for figuring out what to do with your life',
  'text_description': 'If you want 2017 to be an exciting year, design it that way. That\'s the advice of former Google career coach and job strategist Jenny Blake , who has helped more than a thousand people improve their work lives. She recommends creating a "mind map," a visual diagram of your interests and goals. Drawing one doesn\'t take long and could help you figure out the next project, hobby or career change that will make the new year happier and more successful, Blake says. "My favorite way to brainstorm creatively, whether it\'s about values or setting goals for the new year, is through mind maps," Blake tells CNBC. To make one, write down the year at the center of a piece of paper, and then draw spokes with different themes that are important to you. For example, your spokes could be business, personal life, health and fitness, fun, o

    TODO: User Profile Modeling:

    Represent user interests based on their interaction history.
    Extract features from articles using NLP techniques (e.g., topic modeling, embeddings).

    Recommendation Engine:

    Implement collaborative filtering (user-based or item-based) to suggest articles.
    Fine-tune a pre-trained embedding model (such as BERT, Word2Vec, or similar) on your news dataset to improve content representation and matching accuracy.

    Personalization Logic:

    Combine collaborative filtering and fine-tuned embeddings to generate a ranked list of recommended articles for each user.
    Ensure your agent adapts recommendations based on ongoing user interactions (e.g., clicks, skips).
